<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T05-L02-paged-kv-and-continuous-batching/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T05-L02-paged-kv-and-continuous-batching/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T05-L02-paged-kv-and-continuous-batching/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T05-L02-paged-kv-and-continuous-batching/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T05-L02 · nanoserve, part two: a paged KV cache and a continuous-batching scheduler

**You will build:** the two pieces that turn "a model that can decode" into "a server that can
hold many requests at once": a block-table memory manager for the key-value (KV) cache, and a
scheduler that adds and removes requests between every single decoding step. You will serve the
small language model you trained in T03-L03 with them, and measure, against the contiguous cache
and static batching of T05-L01, what they change: throughput, time to first token, and the
memory a cache loses to fragmentation.

**Time:** ~2 hours · **Runs on:** a laptop CPU, no GPU, no download, no network ·
**Prerequisites:** T03-L03-nanolm-cpu-slice, T05-L01-serving-on-a-cpu.

This lesson carries its own copy of the T03-L03 model and trains its own small checkpoint at
the start, because a notebook must run alone in an empty directory. It never loads another
lesson's files.

By the end you will be able to:

1. Implement a block allocator, a block table and paged write and gather that reproduce a
   contiguous cache's contents exactly.
2. Implement a batched decode step over sequences of different lengths through their block
   tables, and verify its logits against recomputing the whole sequence, within a stated tolerance.
3. Implement a continuous-batching scheduler with first-come-first-served admission and
   preemption by recomputation, and verify that every request it serves produces the tokens it
   produces when served alone.
4. Implement exact slot accounting for paged and contiguous caches, and measure throughput,
   time to first token and memory lost to fragmentation for static batching with a contiguous
   cache, continuous batching with a contiguous cache, and continuous batching with a paged
   cache, on one request trace.
5. Measure how block size trades internal fragmentation against block-table size, and how the
   size of the pool and the gap between declared and actual output length decide whether
   paging helps.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T03-L03`, `T05-L01`

**This lesson assumes you can already:**

- write a Python class that keeps a free list of integer ids and a dict from a request id to the list of ids it holds
- gather rows from a numpy array with an integer index array, and write values back into chosen rows by assignment
- compute how many fixed-size blocks n items need with integer ceiling division, and how many slots are left empty

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Efficient Memory Management for Large Language Model Serving with PagedAttention](https://arxiv.org/abs/2309.06180) · Kwon et al. (SOSP 2023, arXiv) · *free* — section 3 on the three sources of KV-cache waste, section 4.2 on the KV cache manager's blocks and block tables, section 4.5 on first-come-first-served scheduling and preemption by swapping or recomputation
- [Orca: A Distributed Serving System for Transformer-Based Generative Models](https://www.usenix.org/conference/osdi22/presentation/yu) · Yu et al. (OSDI 2022) · *free* — the paper that introduced iteration-level scheduling, now called continuous batching; section 3, Challenges and Proposed Solutions, sets out why finished requests should leave the batch at once
- [Operating Systems: Three Easy Pieces, chapter 18, Paging: Introduction](https://pages.cs.wisc.edu/~remzi/OSTEP/vm-paging.pdf) · Remzi and Andrea Arpaci-Dusseau · *free* — fixed-size pages, page tables and why paging avoids external fragmentation, the operating-system idea that PagedAttention borrows for the block table
- [Mastering LLM Techniques: Inference Optimization](https://developer.nvidia.com/blog/mastering-llm-techniques-inference-optimization/) · Shashank Verma and Neal Vaidya (NVIDIA Developer Blog) · *free* — the in-flight batching section: evict finished sequences and admit new requests between iterations, set against static batching
- [Optimization and Tuning, section Preemption (vLLM documentation)](https://docs.vllm.ai/en/latest/configuration/optimization/) · vLLM project · *free* — what a production engine does when the KV cache runs out: preempt a request and recompute it later, rather than swap, and what that costs in latency

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import os
import signal
import sys
import threading
import time
import traceback
from typing import Any, Callable, NamedTuple, Sequence

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__)

SEED = 20260930           # this lesson's one fixed seed: the checkpoint's initial weights come
                          # from their own fixed seed below, every request trace derives from this
LOGIT_TOLERANCE = 1e-9    # "the same logits" means: equal to within this, in float64

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("blocks_needed", "locate"),
    "exercise 2": ("allocate", "release"),
    "exercise 3": ("paged_write", "paged_gather"),
    "exercise 4": ("decode_paged",),
    "exercise 5": ("grow_or_preempt",),
    "exercise 6": ("admit",),
    "exercise 7": ("paged_slots", "contiguous_slots"),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (paged_write, paged_gather)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# How long an exercise's check may run before it counts as stuck, and how long any other cell that runs your
# code (a demonstration, a table) may: a right answer needs a small fraction of either, measured on the worked
# solution (internal/retrofit_guard.py, 10x or more). tools/verify_all.py reads both numbers.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 10


class _Timeout(BaseException):
    """Raised by `_watchdog`. A BaseException, so a student's `except Exception` cannot swallow it."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _watchdog(seconds: float, memory_mib: float = 2048):
    """Stop a block that runs longer than `seconds`, so a loop that never ends becomes a message
    instead of a frozen notebook. It uses SIGALRM, which exists on Linux and macOS (Colab, Kaggle,
    Binder, Codespaces and most laptops) and only in the main thread; where it does not exist the
    block simply runs unguarded. An alarm that was already running (the grader's own time limit)
    keeps its deadline if it is due first and is put back, with the time it had left, afterwards."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_watchdog, "stopped", False):           # a stopped block's frames keep what they built in a
        _watchdog.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = f"still running after {seconds:g} s"
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _Timeout(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _Timeout(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _Timeout(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_watchdog", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _Timeout as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _watchdog.stopped = True
        raise _Timeout(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _watchdog.stopped = True
        raise _Timeout(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = (),
         seconds: float = 0) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A loop that never ends is stopped after
    `seconds` and reported. A demo names the exercises it `needs`: until each has passed its
    check, the demo says which one it is waiting for and skips. Nothing is swallowed: every
    outcome is recorded in `_STATUS` for the progress board at the foot of the notebook, and
    every failure in `_FAILED_CHECKS`, which ends a script run non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _watchdog(seconds or (_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS)):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except _Timeout as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}. That is a loop that never ends: check that every loop "
              "removes or advances something each time round, and that a request you evict or "
              "admit really leaves the list you are walking.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The model you are serving

This is the model from T03-L03, carried over so that the notebook stands alone: a character-level
language model with token and position embeddings, one causal attention head, a residual
connection and a linear head, trained on an excerpt of Shakespeare's Sonnets (public domain, see
`claims.yaml`). Its parameters are the same arrays; the code below is the forward and
backward pass you derived there, given to you here in compact form, plus the pieces a server
needs: **one position at a time** (`embed`, `qkv`, `attend_one`, `readout`) and a **prefill**
that returns a whole prompt's keys and values.

One difference from the earlier lesson is deliberate: the position table is longer, so a request
can be longer. Nothing about paging or scheduling depends on the model, and that is the point of
building them around one you already know.

Read it, then run it. The next cell prints the corpus size and the model's parameter count.

In [ ]:
CORPUS = """From fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou, contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
Making a famine where abundance lies,
Thyself thy foe, to thy sweet self too cruel:
Thou that art now the world's fresh ornament,
And only herald to the gaudy spring,
Within thine own bud buriest thy content,
And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee.

II

When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery so gazed on now,
Will be a tatter'd weed of small worth held:
Then being asked, where all thy beauty lies,
Where all the treasure of thy lusty days;
To say, within thine own deep sunken eyes,
Were an all-eating shame, and thriftless praise.
How much more praise deserv'd thy beauty's use,
If thou couldst answer 'This fair child of mine
Shall sum my count, and make my old excuse,'
Proving his beauty by succession thine!
    This were to be new made when thou art old,
    And see thy blood warm when thou feel'st it cold.

III

Look in thy glass and tell the face thou viewest
Now is the time that face should form another;
Whose fresh repair if now thou not renewest,
Thou dost beguile the world, unbless some mother.
For where is she so fair whose unear'd womb
Disdains the tillage of thy husbandry?
Or who is he so fond will be the tomb,
Of his self-love to stop posterity?
Thou art thy mother's glass and she in thee
Calls back the lovely April of her prime;
So thou through windows of thine age shalt see,
Despite of wrinkles this thy golden time.
    But if thou live, remember'd not to be,
    Die single and thine image dies with thee.

IV

Unthrifty loveliness, why dost thou spend
Upon thyself thy beauty's legacy?
Nature's bequest gives nothing, but doth lend,
And being frank she lends to those are free:
Then, beauteous niggard, why dost thou abuse
The bounteous largess given thee to give?
Profitless usurer, why dost thou use
So great a sum of sums, yet canst not live?
For having traffic with thyself alone,
Thou of thyself thy sweet self dost deceive:
Then how when nature calls thee to be gone,
What acceptable audit canst thou leave?
    Thy unused beauty must be tombed with thee,
    Which, used, lives th' executor to be.

V

Those hours, that with gentle work did frame
The lovely gaze where every eye doth dwell,
Will play the tyrants to the very same
And that unfair which fairly doth excel;
For never-resting time leads summer on
To hideous winter, and confounds him there;
Sap checked with frost, and lusty leaves quite gone,
Beauty o'er-snowed and bareness every where:
Then were not summer's distillation left,
A liquid prisoner pent in walls of glass,
Beauty's effect with beauty were bereft,
Nor it, nor no remembrance what it was:
    But flowers distill'd, though they with winter meet,
    Leese but their show; their substance still lives sweet.


VI

Then let not winter's ragged hand deface,
In thee thy summer, ere thou be distill'd:
Make sweet some vial; treasure thou some place
With beauty's treasure ere it be self-kill'd.
That use is not forbidden usury,
Which happies those that pay the willing loan;
That's for thyself to breed another thee,
Or ten times happier, be it ten for one;
Ten times thyself were happier than thou art,
If ten of thine ten times refigur'd thee:
Then what could death do if thou shouldst depart,
Leaving thee living in posterity?
    Be not self-will'd, for thou art much too fair
    To be death's conquest and make worms thine heir.

VII

Lo! in the orient when the gracious light
Lifts up his burning head, each under eye
Doth homage to his new-appearing sight,
Serving with looks his sacred majesty;
And having climb'd the steep-up heavenly hill,
Resembling strong youth in his middle age,
Yet mortal looks adore his beauty still,
Attending on his golden pilgrimage:
But when from highmost pitch, with weary car,
Like feeble age, he reeleth from the day,
The eyes, 'fore duteous, now converted are
From his low tract, and look another way:
    So thou, thyself outgoing in thy noon:
    Unlook'd, on diest unless thou get a son.

VIII

Music to hear, why hear'st thou music sadly?
Sweets with sweets war not, joy delights in joy:
Why lov'st thou that which thou receiv'st not gladly,
Or else receiv'st with pleasure thine annoy?
If the true concord of well-tuned sounds,
By unions married, do offend thine ear,
They do but sweetly chide thee, who confounds
In singleness the parts that thou shouldst bear.
Mark how one string, sweet husband to another,
Strikes each in each by mutual ordering;
Resembling sire and child and happy mother,
Who, all in one, one pleasing note do sing:
    Whose speechless song being many, seeming one,
    Sings this to thee: 'Thou single wilt prove none.'

IX

Is it for fear to wet a widow's eye,
That thou consum'st thyself in single life?
Ah! if thou issueless shalt hap to die,
The world will wail thee like a makeless wife;
The world will be thy widow and still weep
That thou no form of thee hast left behind,
When every private widow well may keep
By children's eyes, her husband's shape in mind:
Look! what an unthrift in the world doth spend
Shifts but his place, for still the world enjoys it;
But beauty's waste hath in the world an end,
And kept unused the user so destroys it.
    No love toward others in that bosom sits
    That on himself such murd'rous shame commits.

X

For shame! deny that thou bear'st love to any,
Who for thyself art so unprovident.
Grant, if thou wilt, thou art belov'd of many,
But that thou none lov'st is most evident:
For thou art so possess'd with murderous hate,
That 'gainst thyself thou stick'st not to conspire,
Seeking that beauteous roof to ruinate
Which to repair should be thy chief desire.
"""

VOCAB = sorted(set(CORPUS))
VOCAB_SIZE = len(VOCAB)
STOI = {ch: i for i, ch in enumerate(VOCAB)}


def encode(text: str) -> list[int]:
    return [STOI[ch] for ch in text]


def decode(ids) -> str:
    return "".join(VOCAB[int(i)] for i in ids)


DATA = np.array(encode(CORPUS), dtype=np.int64)
_SPLIT = int(0.9 * len(DATA))
TRAIN_DATA, VAL_DATA = DATA[:_SPLIT], DATA[_SPLIT:]

D_MODEL = 48       # width of the residual stream, and of every key and value vector
CONTEXT = 64       # how many positions the model has an embedding for: a request cannot be longer
INIT_SEED = 1234   # the checkpoint's own fixed initialisation seed


def softmax(x: np.ndarray) -> np.ndarray:
    """Softmax over the last axis, shifted by the row maximum so large scores cannot overflow."""
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def init_params(seed: int = INIT_SEED, scale: float = 0.02) -> dict:
    """Fresh float64 parameters from a seeded generator."""
    rng = np.random.default_rng(seed)
    d = D_MODEL
    return {"E": rng.normal(0, scale, (VOCAB_SIZE, d)), "P": rng.normal(0, scale, (CONTEXT, d)),
            "Wq": rng.normal(0, scale, (d, d)), "Wk": rng.normal(0, scale, (d, d)),
            "Wv": rng.normal(0, scale, (d, d)), "Wo": rng.normal(0, scale, (d, d)),
            "W1": rng.normal(0, scale, (d, VOCAB_SIZE)), "b1": np.zeros(VOCAB_SIZE)}


def model_forward(params: dict, inputs: np.ndarray, targets=None):
    """Whole-sequence forward pass, (B, T) token ids in. Returns (loss, cache); loss is None
    without targets. The cache holds every intermediate the backward pass and the server need."""
    b, t = inputs.shape
    if t > CONTEXT:
        raise ValueError(f"{t} positions, but the model has only {CONTEXT} position embeddings")
    x = params["E"][inputs] + params["P"][None, :t]
    q, k, v = x @ params["Wq"], x @ params["Wk"], x @ params["Wv"]
    scores = (q @ k.transpose(0, 2, 1)) / np.sqrt(D_MODEL)
    scores = np.where(np.tril(np.ones((t, t), dtype=bool))[None], scores, -np.inf)
    a = softmax(scores)
    ctx = a @ v
    h = x + ctx @ params["Wo"]
    logits = h @ params["W1"] + params["b1"]
    loss = None
    if targets is not None:
        shifted = logits - logits.max(axis=-1, keepdims=True)
        logp = shifted - np.log(np.exp(shifted).sum(axis=-1, keepdims=True))
        loss = float(-np.take_along_axis(logp, targets[..., None], axis=-1).mean())
    return loss, {"inputs": inputs, "targets": targets, "x": x, "q": q, "k": k, "v": v, "a": a,
                  "ctx": ctx, "h": h, "logits": logits}


def model_backward(params: dict, cache: dict) -> dict:
    """Gradients of the mean cross-entropy with respect to every parameter (T03-L03, section 8-10)."""
    b, t = cache["inputs"].shape
    d = D_MODEL
    onehot = np.zeros_like(cache["logits"])
    np.put_along_axis(onehot, cache["targets"][..., None], 1.0, axis=-1)
    d_logits = (softmax(cache["logits"]) - onehot) / (b * t)
    grads = {"W1": cache["h"].reshape(-1, d).T @ d_logits.reshape(-1, VOCAB_SIZE),
             "b1": d_logits.sum(axis=(0, 1))}
    d_h = d_logits @ params["W1"].T
    grads["Wo"] = cache["ctx"].reshape(-1, d).T @ d_h.reshape(-1, d)
    d_ctx = d_h @ params["Wo"].T
    a, q, k, v, x = cache["a"], cache["q"], cache["k"], cache["v"], cache["x"]
    d_a = d_ctx @ v.transpose(0, 2, 1)
    d_v = a.transpose(0, 2, 1) @ d_ctx
    d_scores = a * (d_a - (d_a * a).sum(axis=-1, keepdims=True)) / np.sqrt(d)
    d_q, d_k = d_scores @ k, d_scores.transpose(0, 2, 1) @ q
    xf = x.reshape(-1, d)
    grads["Wq"], grads["Wk"], grads["Wv"] = (xf.T @ d_q.reshape(-1, d), xf.T @ d_k.reshape(-1, d),
                                             xf.T @ d_v.reshape(-1, d))
    d_x = d_h + d_q @ params["Wq"].T + d_k @ params["Wk"].T + d_v @ params["Wv"].T
    grads["P"] = np.zeros_like(params["P"])
    grads["P"][:t] = d_x.sum(axis=0)
    grads["E"] = np.zeros_like(params["E"])
    np.add.at(grads["E"], cache["inputs"], d_x)
    return grads


def make_batch(data: np.ndarray, batch_size: int, context_len: int, rng):
    """`batch_size` random (input, target) windows; the target is the input shifted by one."""
    starts = rng.integers(0, len(data) - context_len, size=batch_size)
    return (np.stack([data[s:s + context_len] for s in starts]),
            np.stack([data[s + 1:s + context_len + 1] for s in starts]))


def train_checkpoint(steps: int = 2000, lr: float = 3e-3, seed: int = 0, batch_size: int = 32):
    """Train from INIT_SEED with Adam (T03-L03, section 13). Returns (params, [(step, loss)])."""
    params = init_params()
    m = {name: np.zeros_like(p) for name, p in params.items()}
    s = {name: np.zeros_like(p) for name, p in params.items()}
    rng = np.random.default_rng(seed)
    history = []
    for step in range(1, steps + 1):
        inputs, targets = make_batch(TRAIN_DATA, batch_size, CONTEXT, rng)
        loss, cache = model_forward(params, inputs, targets)
        grads = model_backward(params, cache)
        for name, p in params.items():
            m[name] = 0.9 * m[name] + 0.1 * grads[name]
            s[name] = 0.999 * s[name] + 0.001 * grads[name] ** 2
            p -= lr * (m[name] / (1 - 0.9 ** step)) / (np.sqrt(s[name] / (1 - 0.999 ** step)) + 1e-8)
        if step == 1 or step % 500 == 0:
            history.append((step, loss))
    return params, history


def heldout_loss(params: dict, batches: int = 30, seed: int = 5) -> float:
    rng = np.random.default_rng(seed)
    return float(np.mean([model_forward(params, *make_batch(VAL_DATA, 32, CONTEXT, rng))[0]
                          for _ in range(batches)]))


# The model, one position at a time. `x` is (..., D_MODEL): one position, or a batch of them.
def embed(params: dict, tokens, positions) -> np.ndarray:
    """Token embedding plus position embedding."""
    return params["E"][tokens] + params["P"][positions]


def qkv(params: dict, x: np.ndarray):
    """The three projections of `x`: (query, key, value), each shaped like `x`."""
    return x @ params["Wq"], x @ params["Wk"], x @ params["Wv"]


def attend_one(q: np.ndarray, keys: np.ndarray, values: np.ndarray) -> np.ndarray:
    """ONE query vector `q` (D,) against every stored key and value, (T, D) each: softmax(k·q / sqrt(D))
    weights the values. Everything stored is in the query's past, so no mask is needed."""
    return softmax(keys @ q / np.sqrt(q.shape[-1])) @ values


def readout(params: dict, x: np.ndarray, context: np.ndarray) -> np.ndarray:
    """Residual connection, then the linear head: the vocabulary scores (logits)."""
    return (x + context @ params["Wo"]) @ params["W1"] + params["b1"]


def prefill_kv(params: dict, ids: Sequence[int]):
    """Run a whole prompt at once. Returns (logits of its last position, keys (T, D), values (T, D)):
    the keys and values of EVERY position, which is what a cache has to be filled with."""
    _, cache = model_forward(params, np.array([list(ids)], dtype=np.int64))
    return cache["logits"][0, -1], cache["k"][0], cache["v"][0]


def sample_token(logits: np.ndarray, u: float) -> int:
    """Draw a token by inverting the cumulative distribution at `u` in [0, 1). The uniform number
    `u` is supplied by the caller, one per generated token, so which token a request draws depends
    only on the request's own scores and its own `u`, never on who else is in the batch."""
    cdf = np.cumsum(softmax(logits))
    return int(min(np.searchsorted(cdf, u, side="right"), len(cdf) - 1))


_uniform = np.log(VOCAB_SIZE)
_counts = np.bincount(TRAIN_DATA, minlength=VOCAB_SIZE).astype(np.float64)
_p = _counts[_counts > 0] / _counts.sum()
UNIGRAM_ENTROPY = float(-(_p * np.log(_p)).sum())
print(f"corpus: {len(CORPUS):,} characters, {VOCAB_SIZE} distinct; the model has "
      f"{sum(int(a.size) for a in init_params().values()):,} parameters, {CONTEXT} positions")
print(f"baselines, nats per character: uniform {_uniform:.3f}, unigram entropy {UNIGRAM_ENTROPY:.3f}")

### Train the checkpoint

A short Adam run on a laptop CPU. The model is small and so is the corpus,
so it will not write well; it does not need to. What matters for serving is that its
scores are real, uneven and different at every position, so that a serving bug shows up as a
different token, not as an unnoticed rounding of nothing. The cell prints how it did against the
baselines above.

In [ ]:
PARAMS: dict = {}


def _train_and_report() -> None:
    t0 = time.perf_counter()
    params, history = train_checkpoint()
    took = time.perf_counter() - t0
    for step, loss in history:
        print(f"  step {step:5d}  training loss {loss:.4f}")
    PARAMS.update(params)
    held = heldout_loss(params)
    print(f"held-out loss {held:.3f} nats per character, against a unigram entropy of "
          f"{UNIGRAM_ENTROPY:.3f}: the model is {'using' if held < UNIGRAM_ENTROPY else 'not yet using'} "
          f"context")
    print(f"trained in {took:.1f} s on this machine (a timing: yours will differ, nothing else depends on it)")


if __name__ == "__main__":
    _train_and_report()

## 2. Serving one request alone: the contiguous cache

T05-L01 gave every request a slab: one preallocated array with room for the longest the request
could become, filled from the front. That is the reference in this lesson. `generate_alone` serves
one request with it, one token at a time, and is the definition of "the right answer": whatever
the paged, batched, preempted server does to a request, the tokens must equal these.

Requests also arrive as a **trace**: when each one arrives, how long its prompt is, how many
tokens it will really produce, and the **cap** it declared (`max_tokens` in a real API). The
server knows the prompt length and the cap when the request arrives. It does *not* know how many
tokens the request will really produce: the model decides that as it goes. A slab has to be sized
for the cap; that gap between cap and reality is where a contiguous cache loses memory.

In [ ]:
class SlabCache:
    """T05-L01's cache, for this one-layer, one-head model: `max_len` slots, filled from the front."""

    def __init__(self, max_len: int, dim: int = D_MODEL):
        self.k = np.zeros((max_len, dim))
        self.v = np.zeros((max_len, dim))
        self.length = 0
        self.max_len = max_len


def generate_alone(params: dict, prompt: Sequence[int], us: Sequence[float]):
    """Serve ONE request with a contiguous slab: prefill the prompt, then one cached decode step per
    token. `us[j]` is the uniform number that draws token j, so `len(us)` is the number of tokens.
    Returns (tokens, logits): a list of ints, and the (len(us), vocab) scores that chose them."""
    n_new = len(us)
    prompt = list(prompt)
    slab = SlabCache(len(prompt) + n_new)
    logits, keys, values = prefill_kv(params, prompt)
    slab.k[:len(prompt)], slab.v[:len(prompt)], slab.length = keys, values, len(prompt)
    tokens, rows = [], []
    for j in range(n_new):
        rows.append(logits)
        tokens.append(sample_token(logits, us[j]))
        if j < n_new - 1:
            x = embed(params, tokens[-1], slab.length)        # the new token's position is the length
            q, k, v = qkv(params, x)
            slab.k[slab.length], slab.v[slab.length] = k, v
            slab.length += 1
            logits = readout(params, x, attend_one(q, slab.k[:slab.length], slab.v[:slab.length]))
    return tokens, np.array(rows).reshape(n_new, VOCAB_SIZE)


class TraceEntry(NamedTuple):
    """One request of a trace. `new_tokens` is what the request will really produce, and is never
    visible to a scheduler beforehand; `cap` is the maximum it declared, and is."""
    rid: int
    arrival: int       # the iteration at which the request reaches the server
    prompt_len: int
    new_tokens: int
    cap: int


def make_trace(n: int, seed: int, max_gap: int = 2, prompt_range=(8, 64), cap_range=(16, 160),
               run_to_cap: bool = False) -> list:
    """`n` requests, in arrival order. Gaps between arrivals are 0..max_gap iterations. The prompt
    length and the cap are uniform over their ranges. Unless `run_to_cap`, a request really produces a
    uniformly random number of tokens between 1 and its cap (people stop early, models emit a stop
    token); with `run_to_cap` every request uses its whole cap. Given, and seeded."""
    rng = np.random.default_rng(seed)
    arrivals = np.cumsum(rng.integers(0, max_gap + 1, size=n))
    prompts = rng.integers(prompt_range[0], prompt_range[1] + 1, size=n)
    caps = rng.integers(cap_range[0], cap_range[1] + 1, size=n)
    fraction = rng.random(n)          # drawn either way, so both traces share their other columns
    real = caps if run_to_cap else np.minimum(caps, 1 + (fraction * caps).astype(np.int64))
    return [TraceEntry(i, int(arrivals[i]), int(prompts[i]), int(real[i]), int(caps[i]))
            for i in range(n)]


def _show_slab_waste() -> None:
    demo = make_trace(1, SEED)[0]
    reserved = demo.prompt_len + demo.cap - 1        # the last token is never fed back, so never cached
    filled = demo.prompt_len + demo.new_tokens - 1
    print(f"one request: prompt {demo.prompt_len}, declared cap {demo.cap}, really produced {demo.new_tokens}")
    print(f"its slab reserves {reserved} positions for its whole life; it ever fills {filled} of them, "
          f"so {reserved - filled} positions ({100 * (reserved - filled) / reserved:.1f}%) are reserved "
          "and never used, and nobody else can use them while it runs")
    prompt = encode(CORPUS[:demo.prompt_len])
    us = np.random.default_rng(SEED + 1).random(demo.new_tokens)
    tokens, _ = generate_alone(PARAMS, prompt, us)
    print(f"served alone it says: {decode(prompt)!r} -> {decode(tokens)!r}")


if __name__ == "__main__":
    _show_slab_waste()

## 3. Exercise 1: blocks and the block table

Paging cuts the KV cache into **blocks** of `block_size` token slots, all the same size, kept in
one pool. A sequence's cache is a list of block ids, its **block table**: `table[i]` holds the
sequence's logical positions `i * block_size` to `(i + 1) * block_size - 1`, and the blocks can
be anywhere in the pool, in any order. Kwon et al. (see `claims.yaml`) describe it as the
operating system's virtual memory: blocks are pages, tokens are bytes, requests are processes.

Two small functions turn positions into places:

- `blocks_needed(n_tokens, block_size)`: how many blocks hold `n_tokens` positions.
- `locate(table, position, block_size)`: the physical block and the slot inside it that hold a
  logical position. A position outside the table is an error, never a wrap-around.

<details><summary>💡 Hint 1 — what to think about</summary>

How many blocks does a sequence of exactly one block's worth need, and how many when it is one
token longer? What should zero tokens need? For `locate`, which block of the table does a
position fall in, and how far into that block is it? What does Python do with `table[-1]`, and is
that what a negative position should mean?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

The number of blocks is the token count divided by the block size, rounded UP, using whole-number
arithmetic only (a float division is a trap at large counts). For `locate`, the whole-number
quotient and remainder of the position by the block size are the table index and the slot. Check
the position against the table's length yourself before indexing.
</details>

In [ ]:
def blocks_needed(n_tokens: int, block_size: int) -> int:
    """How many blocks it takes to hold `n_tokens` positions when each block holds `block_size`.

    Zero tokens need zero blocks; one token needs a whole block. Raises ValueError for a negative
    `n_tokens` or a `block_size` below 1. Whole-number arithmetic only.

    Worked example: with block_size 4, then 0 tokens -> 0 blocks, 1 -> 1, 4 -> 1, 5 -> 2,
    8 -> 2, 9 -> 3.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def locate(table: Sequence[int], position: int, block_size: int) -> tuple[int, int]:
    """Where logical `position` of a sequence lives: (physical block id, slot inside that block).

    `table[i]` is the physical block that holds logical positions i * block_size up to
    (i + 1) * block_size - 1. Raises IndexError if `position` is negative or falls beyond the
    blocks in `table`; it must never wrap around like `table[-1]` does.

    Worked example: table [7, 2, 9] and block_size 4: position 0 -> (7, 0), position 5 -> (2, 1),
    position 11 -> (9, 3). Position 12 raises IndexError, and so does position -1.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_blocks_and_locate() -> None:
    got = [blocks_needed(n, 4) for n in (0, 1, 4, 5, 8, 9)]
    assert got == [0, 1, 1, 2, 2, 3], (
        f"blocks_needed(n, 4) for n = 0, 1, 4, 5, 8, 9 gave {got}, expected [0, 1, 1, 2, 2, 3]: "
        "round UP, and a sequence that exactly fills its blocks needs no extra one")
    assert type(blocks_needed(10, 3)) is int, "blocks_needed must return a plain int"
    assert blocks_needed(10 ** 18 + 1, 10 ** 9) == 10 ** 9 + 1, (
        "blocks_needed lost precision on a huge count: use whole-number arithmetic, not a float division")
    assert locate([7, 2, 9], 0, 4) == (7, 0) and locate([7, 2, 9], 5, 4) == (2, 1), (
        "locate: position 0 is slot 0 of the FIRST block of the table; position 5 is slot 1 of the second")
    assert locate([7, 2, 9], 11, 4) == (9, 3), "the last position of a full table is the last slot of its last block"
    assert locate([0, 3], 2, 4) == (0, 2), (
        "block id 0 is a real block: a lookup that treats a falsy id as 'missing' is wrong")
    for bad in (12, 13, -1, -5):
        try:
            locate([7, 2, 9], bad, 4)
        except IndexError:
            continue
        raise AssertionError(f"locate(table, {bad}, 4) should raise IndexError: a position past the table's "
                             "blocks, or a negative one, is an error (table[-1] would silently wrap)")
    try:
        one = blocks_needed(3, 1)
    except ValueError:
        raise AssertionError("blocks_needed(3, 1) raised ValueError, but a block size of 1 is valid (one position "
                             "per block): only a block size below 1 is an error") from None
    assert one == 3, f"blocks_needed(3, 1) gave {one}: with one position per block, 3 positions need 3 blocks"
    for args in ((-1, 4), (5, 0)):
        try:
            blocks_needed(*args)
        except ValueError:
            continue
        raise AssertionError(f"blocks_needed{args} should raise ValueError")
    print("blocks_needed and locate: rounding up, exact fills, block id 0, and the two ends of the table")


if __name__ == "__main__":
    _try("exercise 1", _check_blocks_and_locate)

## 4. Exercise 2: the block pool

The pool hands blocks out and takes them back. There is nothing clever about it: a set of free
block ids, and two operations. Real allocators are this simple for the same reason paging is
attractive: every block is the same size, so **any free block can serve any request**, and the
pool cannot fragment. (You will see the opposite case, the contiguous slab, in section 12.)

The lesson fixes two rules so that layouts are reproducible: `allocate` hands out the **lowest**
free ids first, in ascending order, and both operations are all-or-nothing: an error changes
nothing. A double release, or releasing a block that was never handed out, corrupts a real
allocator silently (two requests would then share one block), so `release` refuses it.

<details><summary>💡 Hint 1 — what to think about</summary>

What exactly must be true of the pool before you change it, in each of the two functions? If
something is wrong halfway through a list of ids, what state have you left the pool in? Is
block id 0 special?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

In `allocate`, check the request against the number of free blocks first, then take that many
of the smallest ids and remove them from the set. In `release`, validate the WHOLE list
(in range, currently in use, no repeats) before adding any of it back.
</details>

In [ ]:
class OutOfBlocks(RuntimeError):
    """The pool does not have that many free blocks."""


class BlockPool:
    """A fixed number of equally sized KV blocks, numbered 0 .. n_blocks - 1. Given."""

    def __init__(self, n_blocks: int):
        self.n_blocks = n_blocks
        self.free_ids: set[int] = set(range(n_blocks))

    @property
    def n_free(self) -> int:
        return len(self.free_ids)


def allocate(pool: BlockPool, n: int) -> list[int]:
    """Take `n` blocks from `pool` and return their ids as a list of ints in ascending order,
    always the lowest free ids. `n == 0` returns []. Raises OutOfBlocks, changing nothing, when
    fewer than `n` blocks are free, and ValueError when `n` is negative.

    Worked example: a fresh pool of 6 blocks: allocate(pool, 3) -> [0, 1, 2]. Release [1] and
    allocate(pool, 2) -> [1, 3]: the lowest free ids are 1 and 3.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def release(pool: BlockPool, blocks: Sequence[int]) -> None:
    """Return `blocks` to `pool`. Raises ValueError, changing nothing, if any id is not a block of
    this pool, is already free, or appears more than once in `blocks`. `release(pool, [])` does nothing.

    Worked example: after allocate(pool, 3) -> [0, 1, 2], release(pool, [2, 0]) frees 0 and 2;
    releasing [0] again raises ValueError, and so does releasing [1, 1].
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_pool() -> None:
    pool = BlockPool(6)
    first = allocate(pool, 3)
    assert first == [0, 1, 2] and all(type(b) is int for b in first), (
        f"a fresh pool's first 3 blocks should be [0, 1, 2] as plain ints, got {first!r}: take the LOWEST free ids, "
        "including id 0")
    release(pool, [1])
    assert allocate(pool, 2) == [1, 3], "after releasing block 1, the lowest two free ids are [1, 3]"
    assert pool.n_free == 2 and allocate(pool, 0) == [] and pool.n_free == 2, "allocate(pool, 0) is a no-op returning []"
    before = set(pool.free_ids)
    try:
        allocate(pool, 3)
    except OutOfBlocks:
        pass
    else:
        raise AssertionError("asking for more blocks than are free must raise OutOfBlocks")
    assert pool.free_ids == before, "a refused allocation must leave the pool exactly as it was"
    assert len(allocate(pool, 2)) == 2 and pool.n_free == 0, "asking for exactly what is free succeeds, and empties the pool"
    try:
        allocate(BlockPool(2), -1)
    except ValueError:
        pass
    else:
        raise AssertionError("allocate(pool, -1) should raise ValueError")
    for bad, why in (([0, 0], "an id twice in one call"), ([9], "a block that does not exist"), ([6], "the id one past the last block"),
                     ([-1], "a negative id"), ([4], "a block that is already free")):
        pool2 = BlockPool(6)
        allocate(pool2, 3)
        free_before = set(pool2.free_ids)
        try:
            release(pool2, bad)
        except ValueError:
            assert pool2.free_ids == free_before, f"releasing {bad} raised, but changed the pool first"
        else:
            raise AssertionError(f"release(pool, {bad}) ({why}) should raise ValueError")
    shuffled = BlockPool(200)
    shuffled.free_ids = {70, 5, 130, 12}          # a set does not iterate in ascending order
    got = allocate(shuffled, 2)
    assert got == sorted(got) and set(got) | shuffled.free_ids == {70, 5, 130, 12} and max(got) < min(shuffled.free_ids), (
        f"allocate gave {got} from the free set {{70, 5, 130, 12}}: take the LOWEST free ids in ascending order, "
        "not whichever ids the set happens to iterate first")
    pool3 = BlockPool(4)
    allocate(pool3, 2)
    try:
        release(pool3, [1, 2])          # 1 is in use, 2 is free: the whole call must be refused
    except ValueError:
        pass
    assert pool3.free_ids == {2, 3}, "release([1, 2]): 2 is already free, so the call fails and block 1 must NOT be freed"
    print("allocate and release: lowest ids first, all-or-nothing errors, block 0, exact fit")


if __name__ == "__main__":
    _try("exercise 2", _check_pool)

## 5. Exercise 3: writing and reading through a block table

The KV blocks live in one array pair, `PagedKV.k` and `.v`, each `(n_blocks, block_size, D_MODEL)`:
block `b`, slot `s` holds one position's key (or value) vector. Two functions use a block table
to reach it.

- `paged_write(cache, table, position, k, v)` stores one position's key and value where the
  table says that position lives, and touches nothing else.
- `paged_gather(cache, table, length)` reads positions `0 .. length - 1` back, in LOGICAL order,
  as two `(length, D_MODEL)` arrays: what attention needs, wherever the blocks happen to be.
  It returns only `length` positions. The rest of the last block is not part of the sequence, and
  after blocks have been reused it may hold another request's old numbers.

A position past the table's blocks is an `IndexError`, as in `locate`. Use `locate`.

<details><summary>💡 Hint 1 — what to think about</summary>

Which two numbers identify a slot in the cache array? Where have you already written the
function that turns a position into them? For gathering, would it be simplest to fetch positions
one at a time or a block at a time, and what must the last, partly used block contribute?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Write: look the position up, then assign the key vector and the value vector into that block and
slot. Gather: check the length fits the table, make output arrays of the right shape, and fill
each logical position from its own slot. Reading whole blocks and cutting the result to `length`
also works, if you cut it.
</details>

In [ ]:
class PagedKV:
    """The KV block store: `k[b, s]` and `v[b, s]` are the key and value vectors in slot `s` of
    physical block `b`. Given."""

    def __init__(self, n_blocks: int, block_size: int, dim: int = D_MODEL, dtype=np.float64):
        self.block_size = block_size
        self.k = np.zeros((n_blocks, block_size, dim), dtype=dtype)
        self.v = np.zeros((n_blocks, block_size, dim), dtype=dtype)


def paged_write(cache: PagedKV, table: Sequence[int], position: int, k: np.ndarray, v: np.ndarray) -> None:
    """Store the key `k` and value `v` (each `(D_MODEL,)`) of logical `position` where `table` says
    it lives. Writes exactly one slot of `cache.k` and one of `cache.v`. Raises IndexError, having
    written nothing, when the table has no place for `position`.

    Worked example: block_size 4, table [7, 2]: position 5 is slot 1 of block 2, so
    cache.k[2, 1] becomes `k` and cache.v[2, 1] becomes `v`, and nothing else changes.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def paged_gather(cache: PagedKV, table: Sequence[int], length: int):
    """The first `length` positions of the sequence whose blocks are `table`, in logical order:
    returns (keys, values), each of shape (length, D_MODEL). `length == 0` gives two empty
    (0, D_MODEL) arrays. Raises IndexError when `length` is negative or larger than the table can hold.

    Worked example: block_size 4, table [7, 2], length 6: positions 0-3 come from block 7 and
    positions 4-5 from slots 0 and 1 of block 2; the other two slots of block 2 are NOT returned.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_paged_write_gather() -> None:
    rng = np.random.default_rng(0)
    cache = PagedKV(n_blocks=6, block_size=4, dim=3)
    table = [5, 0, 3]                       # scattered, and including block 0
    ks, vs = rng.normal(size=(10, 3)), rng.normal(size=(10, 3))
    for i in range(10):
        paged_write(cache, table, i, ks[i], vs[i])
    assert np.array_equal(cache.k[0, 1], ks[5]) and np.array_equal(cache.v[3, 1], vs[9]), (
        "position 5 belongs in slot 1 of the SECOND block of the table (block 0 here), and position 9 in "
        "slot 1 of the third (block 3): look the position up with locate()")
    keys, values = paged_gather(cache, table, 10)
    assert keys.shape == (10, 3) and np.array_equal(keys, ks) and np.array_equal(values, vs), (
        "gathering all ten positions did not return what was written, in logical order: the table's order, "
        "not the blocks' numeric order, is the sequence's order")
    for n in (0, 1, 3, 4, 5, 8):
        gk, gv = paged_gather(cache, table, n)
        assert gk.shape == (n, 3) and gv.shape == (n, 3) and np.array_equal(gk, ks[:n]) and np.array_equal(gv, vs[:n]), (
            f"paged_gather(..., {n}) should return exactly the first {n} positions, shaped ({n}, 3): "
            "return `length` positions, not whole blocks")
    assert np.all(cache.k[5, :] == ks[:4]) and not np.any(cache.k[1]) and not np.any(cache.k[2]) and not np.any(cache.k[4]), (
        "a write must touch only its own slot: blocks not in the table changed")
    cache.k[3, 2:] = 99.0                    # the leftovers of some earlier request, past position 9
    assert np.array_equal(paged_gather(cache, table, 10)[0], ks), (
        "the unused tail of the last block leaked into the result: return only `length` positions")
    try:
        full = paged_gather(cache, table, 12)[0]
    except IndexError:
        raise AssertionError("paged_gather(cache, table, 12) raised IndexError, but 12 is exactly the table's 12 slots, "
                             "which is allowed: only a length beyond the table's capacity is an error") from None
    assert full.shape == (12, 3), f"paged_gather(cache, table, 12) gave shape {full.shape}, expected (12, 3)"
    for bad in (13, 100):
        try:
            paged_gather(cache, table, bad)
        except IndexError:
            continue
        raise AssertionError(f"paged_gather(cache, table, {bad}) asks for more positions than the table's 12 slots: IndexError")
    try:
        paged_gather(cache, table, -1)
    except IndexError:
        pass
    except Exception as exc:
        raise AssertionError(f"paged_gather(cache, table, -1) raised {type(exc).__name__}, expected IndexError: a negative "
                             "length is refused like a length beyond the table, before anything is built") from None
    else:
        raise AssertionError("paged_gather(cache, table, -1) should raise IndexError: a negative length is an error")
    snapshot = cache.k.copy()
    try:
        paged_write(cache, table, 12, ks[0], vs[0])
    except IndexError:
        assert np.array_equal(cache.k, snapshot), "a refused write must change nothing"
    else:
        raise AssertionError("paged_write at position 12 of a 12-slot table should raise IndexError")
    print("paged_write and paged_gather: scattered blocks, block 0, exact lengths, and a tail that stays hidden")


if __name__ == "__main__":
    _try("exercise 3", _check_paged_write_gather, needs=("exercise 1",))

## 6. Exercise 4: one decoding step for a whole ragged batch

T05-L01's static batch needed every prompt padded to one length, a mask for the padding and
position ids that skipped it. A paged cache needs none of that. Each sequence has its own block
table and its own length, and attention for a row simply reads that row's positions and no
others: nothing to pad, nothing to mask. Sequences of different lengths share one decoding step
because the *step* is the batch, not the arrays.

`decode_paged(params, cache, tables, positions, tokens)` advances every row by one token. For
each row: embed its new token at its position, project it (`qkv`), **write** its key and value
into the cache through the row's table at that position, **gather** the row's whole history, the
new position included, let the single query attend over it (`attend_one`), and finish (`readout`).
`positions[i]` is the new token's position id, which is also how many positions row `i` had
cached before. Given: `embed`, `qkv`, `attend_one`, `readout` (section 1), and your exercises
1 to 3.

<details><summary>💡 Hint 1 — what to think about</summary>

In what order must the write and the gather happen, so that the new token can attend to itself?
How many positions should the gather return? What does each row need that another row must not
be allowed to touch, and what would go wrong if you assumed the blocks were consecutive?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Loop over the rows. For each: embed, project, write at its position, gather one more position
than it had before, attend with the query against those keys and values, and read out one row of
logits. Collect the rows into a `(B, VOCAB_SIZE)` array; an empty batch gives shape `(0, VOCAB_SIZE)`.
</details>

In [ ]:
def decode_paged(params: dict, cache: PagedKV, tables: Sequence[Sequence[int]], positions, tokens) -> np.ndarray:
    """Advance every row by ONE token through its own block table. Returns (B, VOCAB_SIZE) logits.

    tables: one block table per row.  positions: (B,) the position id of each row's new token
    (= how many positions the row already has in the cache).  tokens: (B,) the new token ids.
    Each row's new key and value are written into the cache; its history is read back through its
    table; rows never see each other's blocks. B may be 0.

    Worked example: two rows with 5 and 9 cached positions (block_size 4) get positions [5, 9].
    Row 0 attends over 6 positions (its 5 and itself), row 1 over 10: no padding, no mask.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def prefill_paged(params: dict, cache: PagedKV, table: Sequence[int], ids: Sequence[int]) -> np.ndarray:
    """Fill a sequence's blocks with the keys and values of `ids` (a whole prompt, one forward pass)
    and return the logits of its last position. Uses your `paged_write`. Given."""
    logits, keys, values = prefill_kv(params, ids)
    for position in range(len(ids)):
        paged_write(cache, table, position, keys[position], values[position])
    return logits


def _toy_params() -> dict:
    """Random but not-tiny weights, so that every position's scores differ visibly. For the checks."""
    return init_params(seed=7, scale=0.5)


def _check_decode_paged() -> None:
    params = _toy_params()
    block_size = 4
    cache = PagedKV(n_blocks=12, block_size=block_size)
    cache.k[:], cache.v[:] = 7.0, 7.0                      # stale numbers from earlier tenants
    tables = [[9, 2, 7], [0, 11], [5, 10, 1]]              # scattered, out of order, block 0 included
    lengths = [9, 5, 3]                                    # cached positions per row: not sorted, not equal
    seqs = [_make_ids(n, 30 + i) for i, n in enumerate(lengths)]
    for table, seq in zip(tables, seqs):
        prefill_paged(params, cache, table, seq)
    new_tokens = np.array([11, 3, 20])
    got = decode_paged(params, cache, tables, np.array(lengths), new_tokens)
    assert got.shape == (3, VOCAB_SIZE), f"returned shape {got.shape}, expected (3, {VOCAB_SIZE})"
    for i in range(3):
        _, full = model_forward(params, np.array([seqs[i] + [int(new_tokens[i])]]))
        want = full["logits"][0, -1]
        assert np.max(np.abs(got[i] - want)) < LOGIT_TOLERANCE, (
            f"row {i} differs from recomputing its whole sequence by {np.max(np.abs(got[i] - want)):.1e}: check "
            "the new key and value are written BEFORE the gather, that the row's own table and position are used, "
            "and that the gather returns position + 1 entries (the new position included)")
    for i in range(3):
        keys, _ = paged_gather(cache, tables[i], lengths[i] + 1)
        assert np.max(np.abs(keys[-1] - qkv(params, embed(params, int(new_tokens[i]), lengths[i]))[1])) < 1e-12, (
            f"row {i}: the new token's key is not stored at its position: write it through the row's table")
    assert decode_paged(params, cache, [], np.array([], dtype=np.int64), np.array([], dtype=np.int64)).shape == (0, VOCAB_SIZE), (
        "an empty batch should give an array of shape (0, VOCAB_SIZE)")
    print("decode_paged: three rows of different lengths, scattered blocks and stale slots agree with recomputation")


def _make_ids(n: int, seed: int) -> list[int]:
    """A seeded list of token ids. Given."""
    return np.random.default_rng(seed).integers(0, VOCAB_SIZE, n).tolist()


if __name__ == "__main__":
    _try("exercise 4", _check_decode_paged, needs=("exercise 1", "exercise 3"))

In [ ]:
def _paged_equals_alone() -> None:
    """One request decoded through a scattered block table, against `generate_alone`."""
    prompt = encode(CORPUS[40:52])
    us = np.random.default_rng(SEED + 2).random(30)
    want, want_rows = generate_alone(PARAMS, prompt, us)
    block_size = 4
    n_blocks_needed = blocks_needed(len(prompt) + len(us), block_size)
    pool = BlockPool(3 * n_blocks_needed)
    decoys = allocate(pool, 3 * n_blocks_needed)                 # take everything ...
    release(pool, decoys[::3])                                   # ... give back every third block
    table = allocate(pool, n_blocks_needed)
    cache = PagedKV(len(decoys), block_size)
    logits = prefill_paged(PARAMS, cache, table, prompt)
    tokens, rows = [], []
    for j in range(len(us)):
        rows.append(logits)
        tokens.append(sample_token(logits, us[j]))
        if j < len(us) - 1:
            logits = decode_paged(PARAMS, cache, [table], np.array([len(prompt) + j]), np.array([tokens[-1]]))[0]
    gap_ok = bool(np.max(np.abs(np.array(rows) - want_rows)) < LOGIT_TOLERANCE)
    print(f"block table {table}: physical blocks are scattered, logical order is the table's order")
    print(f"tokens identical to the contiguous slab's: {tokens == want}; logits equal within "
          f"{LOGIT_TOLERANCE:.0e}: {gap_ok}")
    print(f"served: {decode(prompt)!r} -> {decode(tokens)!r}")


if __name__ == "__main__":
    _try("paged equals alone", _paged_equals_alone, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 7. Continuous batching: what the scheduler decides, every iteration

T05-L01's static batch was fixed for its whole life: a request that finished early kept its row
until the longest one was done, and a request that arrived a moment late waited for the whole
batch. Orca's **iteration-level scheduling** (Yu et al., see `claims.yaml`) is the fix, and it is
only a change of when you decide: instead of once per batch, before **every model iteration**.

An *iteration* here is one forward pass of the whole running batch. Requests just admitted run
their **prefill** (all their known tokens in one pass); requests already running each decode one
token. Every running request produces exactly one token per iteration, and a token produced in
iteration `now` appears at time `now + 1`. So at the start of each iteration the scheduler makes
two kinds of decision, and both are about blocks:

1. **Growth.** A running request that has just filled its last block needs one more for the
   token it is about to append. If the pool has none, somebody has to give one up: the scheduler
   **preempts**. Following Kwon et al. (`claims.yaml`), it serves requests first-come-first-served,
   so the request that arrived earliest is protected and the most recently admitted is evicted.
   Its blocks are freed and it goes back to the FRONT of the waiting line; when it is admitted
   again its cache is rebuilt by **recomputation**: the prompt and the tokens it had already
   produced are prefilled as one longer prompt.
2. **Admission.** The request at the head of the waiting line is admitted if it has arrived,
   there is a free running slot, and the pool holds the blocks its prefill needs. First come,
   first served means a request that does not fit holds up everybody behind it, even a small
   one that would fit.

You write those two decisions. The state they act on, and the loop that runs an iteration around
them, are given. A request holds `blocks_needed(prompt_len + generated)` blocks during the
iteration in which it has `generated` tokens so far: its prompt, its tokens so far, and room for
the one it is about to append.

In [ ]:
class Request:
    """A request in flight. `new_tokens` is how many tokens it will really produce: while a scheduler
    runs, reading it raises, because a real server cannot know it. Given."""

    sealed = False   # set while a scheduling decision runs

    def __init__(self, rid: int, arrival: int, prompt_len: int, new_tokens: int, cap: int):
        self.rid, self.arrival, self.prompt_len, self.cap = rid, arrival, prompt_len, cap
        self._new_tokens = new_tokens
        self.generated = 0              # tokens produced so far (survives preemption)
        self.table: list[int] = []      # the blocks it holds right now, in logical order
        self.first_token_at = None      # the time its first token appeared, once
        self.finish_at = None
        self.preemptions = 0
        self.recomputed = 0             # positions it had to prefill again after being evicted

    @property
    def new_tokens(self) -> int:
        if Request.sealed:
            raise AssertionError("the scheduler read `new_tokens`: a real server cannot know how many "
                                 "tokens a request will produce. Decide from prompt_len, generated and table.")
        return self._new_tokens


@contextlib.contextmanager
def _sealed():
    Request.sealed = True
    try:
        yield
    finally:
        Request.sealed = False


class SchedState:
    """Everything a scheduling decision reads and changes. Given.

    waiting: requests not running, in service order (arrival order; an evicted request goes to the
             front). running: requests holding blocks, OLDEST ADMISSION FIRST.
    pool: the free blocks. block_size: token slots per block. max_running: the most requests
    that may run at once (the batch-size limit of the engine)."""

    def __init__(self, requests: Sequence[Request], n_blocks: int, block_size: int, max_running: int,
                 check_fit: bool = True):
        self.pool = BlockPool(n_blocks)
        self.block_size, self.max_running = block_size, max_running
        self.waiting: list[Request] = sorted(requests, key=lambda r: (r.arrival, r.rid))
        self.running: list[Request] = []
        if check_fit:
            for r in requests:      # a request that could never fit alone would be evicted forever
                if blocks_needed(r.prompt_len + r._new_tokens - 1, block_size) > n_blocks:
                    raise ValueError(f"request {r.rid} needs more blocks than the whole pool has")


def preempt(state: SchedState, victim: Request) -> None:
    """Evict a running request by recomputation: free all its blocks, forget its table, count it,
    remember how many positions it will have to prefill again, and put it at the FRONT of the waiting
    line. Its progress (`generated`) is kept. Given; your scheduler decides WHEN and WHOM."""
    release(state.pool, victim.table)
    victim.table = []
    victim.preemptions += 1
    victim.recomputed += victim.prompt_len + victim.generated
    state.running.remove(victim)
    state.waiting.insert(0, victim)


def advance(state: SchedState, now: int) -> list:
    """After the scheduler has decided: every running request produces one token, appearing at time
    `now + 1`. A request that has produced all its tokens finishes and frees its blocks.
    Returns the requests that finished. Given."""
    done = []
    for r in state.running:
        r.generated += 1
        if r.first_token_at is None:
            r.first_token_at = now + 1
        if r.generated == r.new_tokens:
            r.finish_at = now + 1
            done.append(r)
    for r in done:
        release(state.pool, r.table)
        r.table = []
        state.running.remove(r)
    return done

## 8. Exercise 5: growth, and who gets evicted

`grow_or_preempt(state)` is the first half of an iteration's decisions. Visit the running
requests **oldest first** (the order of `state.running`). A request that will hold
`blocks_needed(prompt_len + generated, block_size)` blocks this iteration but holds fewer takes the
difference from the pool and appends the new blocks to its table. If the pool cannot give it,
evict the **newest** running request (the LAST in `state.running`) with the given `preempt`, and
try again; keep evicting until it fits. If the request being served is itself the newest, it is the one
evicted, and it gets nothing. It returns the ids of the evicted requests, in the order they were evicted.

Two things the scheduler must not do: read `new_tokens` (it raises: nobody knows it), and evict
an older request to help a newer one.

<details><summary>💡 Hint 1 — what to think about</summary>

Whom are you serving, and whom are you allowed to sacrifice, if the pool is short? The list you are
walking changes while you walk it: what happens to a request that was evicted before its turn came?
What does a request that needs no new block do? Could a wrong order or a wrong victim make the same
request lose its blocks again and again?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Walk over a COPY of the running list, oldest first, and skip anything no longer running. For each
request work out how many more blocks it needs. While the pool has fewer free than that, evict the
last request in `state.running` and note its id; if it was the one you were serving, stop with it.
Otherwise (the loop ended because there is now enough room) allocate the blocks and extend its table.
</details>

In [ ]:
def grow_or_preempt(state: SchedState) -> list[int]:
    """Give every running request the blocks it needs this iteration, evicting the newest running
    requests if the pool runs short. Returns the ids evicted, in eviction order.

    Read only `state.running`, `state.pool`, `state.block_size` and each request's `prompt_len`,
    `generated` and `table`. Change tables and the pool; evict only through `preempt`.

    Worked example: block_size 4, running [A, B, C] (A oldest). A and B are each about to append a
    token into a fresh block, and one block is free. A takes it. B finds the pool empty, so C (the
    newest) is evicted; C's blocks are freed, and B takes one of them. Returns [C's id]. If C were
    the only one needing a block and none were free, C would evict itself: returns [C's id] too.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _make_state(n_blocks: int, block_size: int, max_running: int, running=(), waiting=()) -> SchedState:
    """A hand-built state for the checks: running = [(rid, prompt_len, generated, table)], waiting =
    [(rid, arrival, prompt_len, generated)]. The tables are taken out of the pool by hand."""
    state = SchedState([], n_blocks, block_size, max_running, check_fit=False)
    for rid, prompt_len, generated, table in running:
        r = Request(rid, 0, prompt_len, 10 ** 6, 10 ** 6)
        r.generated, r.table = generated, list(table)
        state.running.append(r)
        state.pool.free_ids.difference_update(table)
    for rid, arrival, prompt_len, generated in waiting:
        r = Request(rid, arrival, prompt_len, 10 ** 6, 10 ** 6)
        r.generated = generated
        state.waiting.append(r)
    return state


def _check_grow_or_preempt() -> None:
    # Three running requests, one free block: A grows, B must evict C (the newest) to grow.
    state = _make_state(4, 4, 8, running=[(10, 4, 1, [0]), (11, 4, 1, [1]), (12, 2, 1, [2])],
                        waiting=[(20, 0, 4, 0)])
    evicted = _sealed_call(grow_or_preempt, state)
    assert evicted == [12], (
        f"evicted {evicted}, expected [12]: A takes the one free block, B then finds the pool empty and the "
        "NEWEST running request (12) is the one to go, never an older one")
    tables = {r.rid: r.table for r in state.running}
    assert tables == {10: [0, 3], 11: [1, 2]}, (
        f"tables are {tables}: the evicted request's block (2) is freed and given to B, and each grown table "
        "gets the new block appended at its end")
    assert [r.rid for r in state.waiting] == [12, 20] and state.waiting[0].table == [] and state.waiting[0].preemptions == 1, (
        f"the waiting line is {[r.rid for r in state.waiting]}: the evicted request must go to the FRONT of the line, ahead of "
        "request 20 that was already waiting, with an empty table, counted once (evict with the given preempt)")
    # The newest request itself needs a block and nothing is free: it evicts itself.
    state = _make_state(3, 4, 8, running=[(1, 4, 1, [0]), (2, 4, 1, [1])])
    evicted = _sealed_call(grow_or_preempt, state)
    assert evicted == [2] and [r.rid for r in state.running] == [1] and state.running[0].table == [0, 2] and \
        state.pool.free_ids == {1}, (
        f"evicted {evicted}, running {[r.rid for r in state.running]}: request 1 (older) takes the free block; "
        "request 2 then needs one, is the newest, and is evicted itself, freeing its block")
    # Nothing needs a block, and the pool is empty: nothing may happen.
    state = _make_state(2, 4, 8, running=[(1, 3, 1, [0]), (2, 3, 1, [1])])
    assert _sealed_call(grow_or_preempt, state) == [] and state.pool.n_free == 0 and len(state.running) == 2, (
        "a request that still has room in its last block needs nothing: do not evict when nobody needs a block")
    print("grow_or_preempt: oldest served first, the newest evicted, self-eviction, and no eviction when none is needed")


def _sealed_call(fn, *args):
    """Call a scheduling function with `new_tokens` sealed, as the real loop does."""
    with _sealed():
        return fn(*args)


if __name__ == "__main__":
    _try("exercise 5", _check_grow_or_preempt, needs=("exercise 1", "exercise 2"))

## 9. Exercise 6: admission

`admit(state, now)` is the second half. While the waiting line is not empty and fewer than
`max_running` requests are running, look at the request at the **head** of the line:

- if it has not arrived yet (`arrival > now`), stop;
- if the pool has fewer free blocks than its prefill needs, `blocks_needed(prompt_len + generated,
  block_size)`, stop. Do NOT skip over it to a smaller request behind it: first come, first served;
- otherwise take those blocks from the pool as its table, move it from the front of `waiting` to the
  END of `running`, and go round again.

It returns the ids of the requests it admitted, in order. A request that was evicted earlier is
admitted the same way: its `generated` tokens count towards the blocks it needs.

<details><summary>💡 Hint 1 — what to think about</summary>

Which of the three conditions ends the whole loop, and which only decides about one request? Is a
request that arrives at exactly time `now` here yet? What is different about the number of blocks a
resumed request needs, compared with a fresh one?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Loop while there is a head of the line and room in `running`. Compute the head's block need from its
prompt length AND its progress; compare with the free blocks; if it does not fit or has not
arrived, leave the loop. Otherwise allocate, set its table, and move it between the two lists,
recording its id. Make sure every pass of the loop removes something from `waiting`, or ends it.
</details>

In [ ]:
def admit(state: SchedState, now: int) -> list[int]:
    """Admit requests from the head of the waiting line, first come first served. Returns their ids.

    Stops at the first head that has not arrived (`arrival > now`), does not fit the free blocks, or
    when `max_running` requests are running. Admitted requests get their blocks, leave `waiting`, and
    are appended to `running`. Read only `prompt_len`, `generated`, `arrival` and the state.

    Worked example: block_size 4, 5 free blocks, waiting [X needs 2 blocks, Y needs 4, Z needs 1], all
    arrived. X is admitted (3 left); Y does not fit, so the scheduler stops: Z waits behind Y even though
    it would fit. Returns [X's id].
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_admit() -> None:
    state = _make_state(5, 4, 8, waiting=[(1, 0, 6, 0), (2, 0, 13, 0), (3, 0, 1, 0)])
    got = _sealed_call(admit, state, 0)
    assert got == [1], (
        f"admitted {got}, expected [1]: request 1 needs 2 blocks and fits; request 2 needs 4 with only 3 left, "
        "and the line must stop there: request 3 would fit but it is BEHIND request 2")
    assert [r.rid for r in state.waiting] == [2, 3] and state.running[0].table == [0, 1] and state.pool.n_free == 3, (
        "an admitted request leaves the waiting line, holds the lowest free blocks, and the pool shrinks by them")
    exact = _make_state(6, 4, 8, waiting=[(1, 0, 6, 0), (2, 0, 5, 0), (3, 0, 8, 0)])
    assert _sealed_call(admit, exact, 0) == [1, 2, 3] and exact.pool.n_free == 0, (
        "three requests needing 2 blocks each exactly fill a pool of 6: admit keeps going while the head fits "
        "(free == need is a fit), and stops only when it does not")
    later = _make_state(4, 4, 8, waiting=[(7, 5, 3, 0)])
    assert [_sealed_call(admit, later, t) for t in (4, 5)] == [[], [7]], (
        "a request is admitted from the iteration `now == arrival` on, and not before")
    crowded = _make_state(8, 4, 2, running=[(1, 3, 1, [0])], waiting=[(2, 0, 3, 0), (3, 0, 3, 0)])
    assert _sealed_call(admit, crowded, 0) == [2] and [r.rid for r in crowded.running] == [1, 2], (
        "max_running is a limit on how many run at once: with one slot left only the head is admitted")
    resumed = _make_state(6, 4, 8, waiting=[(4, 0, 6, 5)])
    _sealed_call(admit, resumed, 3)
    assert len(resumed.running[0].table) == 3, (
        f"a request evicted after 5 tokens has 6 + 5 positions to rebuild, which is 3 blocks, not 2: "
        f"its table has {len(resumed.running[0].table)}")
    print("admit: first come first served, arrival time, the running limit, and resumed requests")


if __name__ == "__main__":
    _try("exercise 6", _check_admit, needs=("exercise 1", "exercise 2"))


def plan_iteration(state: SchedState, now: int):
    """One iteration's scheduling decisions. Returns (evicted ids, admitted ids). Given.

    Growth first. Nobody is admitted in an iteration that had to evict anyone: the pool is short,
    and the evicted request would only be readmitted straight away."""
    with _sealed():
        evicted = grow_or_preempt(state)
        admitted = [] if evicted else admit(state, now)
    return evicted, admitted

## 10. The whole engine, on real requests

Everything now exists to serve real text. `serve_paged` is the loop a real engine runs, and it is
given: each iteration it asks `plan_iteration` (your exercises 5 and 6) who runs, prefills the newly
admitted requests (`prefill_paged`, your exercise 3), decodes the rest as one ragged batch
(`decode_paged`, your exercise 4), draws each request's next token from its own scores with its own
random number, and lets `advance` finish requests and free their blocks.

The pool is deliberately small for the number of requests, to give the scheduler reasons to
preempt some of them and rebuild their caches by recomputation; the output says how often it did. The test is the one that matters
for a serving engine: **does every request produce exactly the tokens it produces when it is served
alone, with a contiguous slab, on an idle machine?** Being evicted, recomputed, batched with strangers and
scattered across blocks must change speed and memory, and never a token.

In [ ]:
ENGINE_BLOCK_SIZE, ENGINE_BLOCKS, ENGINE_MAX_RUNNING = 4, 16, 5


def make_engine_requests(n: int, seed: int) -> list:
    """`n` requests whose prompts are slices of the corpus. Each carries its own uniform numbers `us`
    (one per token it will produce), so its output depends on nothing but itself. Given."""
    rng = np.random.default_rng(seed)
    arrivals = np.cumsum(rng.integers(0, 3, size=n))
    prompt_lens = rng.integers(6, 21, size=n)
    new_tokens = rng.integers(8, 31, size=n)
    starts = rng.integers(0, len(CORPUS) - 20, size=n)
    requests = []
    for i in range(n):
        r = Request(i, int(arrivals[i]), int(prompt_lens[i]), int(new_tokens[i]), int(new_tokens[i]))
        r.prompt = encode(CORPUS[int(starts[i]):int(starts[i]) + int(prompt_lens[i])])
        r.us = np.random.default_rng([seed, i]).random(int(new_tokens[i]))
        r.tokens, r.rows = [], [None] * int(new_tokens[i])
        requests.append(r)
    return requests


def _ensure_progress(state: SchedState, now: int, evicted: list) -> None:
    """Fail loudly, with a message, when a scheduler makes no progress. Given."""
    if not state.running and not evicted:
        head = state.waiting[0]
        raise AssertionError(
            f"nothing is running and nothing was admitted at iteration {now}, although request {head.rid} has "
            f"arrived and the pool has {state.pool.n_free} free blocks of {state.pool.n_blocks}: admit() must admit "
            "the head of the line when it has arrived, fits, and a running slot is free")


def serve_paged(params: dict, requests: list, n_blocks: int, block_size: int, max_running: int):
    """Serve `requests` with the paged cache, the scheduler and the model. Given.
    Returns (iterations, peak blocks in use). Every request ends with `.tokens` and `.rows`."""
    state = SchedState(requests, n_blocks, block_size, max_running)
    cache = PagedKV(n_blocks, block_size)
    now = iterations = peak = 0
    limit = 8 * sum(r._new_tokens for r in requests) + max(r.arrival for r in requests) + 1000
    while state.waiting or state.running:
        if not state.running and state.waiting[0].arrival > now:
            now = state.waiting[0].arrival                  # idle: jump to the next arrival
        evicted, admitted = plan_iteration(state, now)
        _ensure_progress(state, now, evicted)
        peak = max(peak, n_blocks - state.pool.n_free)
        fresh = set(admitted)
        decoding = [r for r in state.running if r.rid not in fresh]
        if decoding:
            logits = decode_paged(params, cache, [r.table for r in decoding],
                                  np.array([r.prompt_len + r.generated - 1 for r in decoding]),
                                  np.array([r.tokens[-1] for r in decoding]))
            for r, row in zip(decoding, logits):
                r.rows[r.generated] = row
                r.tokens.append(sample_token(row, r.us[r.generated]))
        for r in state.running:
            if r.rid in fresh:                              # prefill: the prompt AND anything produced before an eviction
                row = prefill_paged(params, cache, r.table, r.prompt + r.tokens)
                r.rows[r.generated] = row
                r.tokens.append(sample_token(row, r.us[r.generated]))
        advance(state, now)
        now += 1
        iterations += 1
        if iterations > limit:
            raise AssertionError(f"still serving after {iterations} iterations: requests are being evicted "
                                 "and readmitted without finishing (check WHOM grow_or_preempt evicts)")
    return iterations, peak


def _show_engine() -> None:
    requests = make_engine_requests(12, SEED + 3)
    iterations, peak = serve_paged(PARAMS, requests, ENGINE_BLOCKS, ENGINE_BLOCK_SIZE, ENGINE_MAX_RUNNING)
    same_tokens = same_logits = 0
    for r in requests:
        tokens, rows = generate_alone(PARAMS, r.prompt, r.us)
        same_tokens += int(r.tokens == tokens)
        same_logits += int(bool(np.max(np.abs(np.array(r.rows) - rows)) < LOGIT_TOLERANCE))
    preempted = [r for r in requests if r.preemptions]
    print(f"{len(requests)} requests, {ENGINE_BLOCKS} blocks of {ENGINE_BLOCK_SIZE} slots, at most "
          f"{ENGINE_MAX_RUNNING} running: served in {iterations} iterations, peak {peak} blocks in use")
    print(f"{len(preempted)} requests were evicted at least once ({sum(r.preemptions for r in requests)} "
          f"evictions, {sum(r.recomputed for r in requests)} positions prefilled again)")
    print(f"tokens identical to serving alone: {same_tokens} of {len(requests)} requests; logits within "
          f"{LOGIT_TOLERANCE:.0e} on every step: {same_logits} of {len(requests)}")
    show = preempted[0] if preempted else requests[0]
    print(f"request {show.rid} ({'evicted ' + str(show.preemptions) + ' time(s)' if show.preemptions else 'never evicted'}): "
          f"{decode(show.prompt)!r} -> {decode(show.tokens)!r}")
    assert same_tokens == len(requests) and same_logits == len(requests), (
        f"{len(requests) - same_tokens} requests came back different from serving alone: a request must never "
        "notice the others. Suspects: a block table that leaks between rows, a position id that is not the "
        "request's own, a gather that returns too many or too few positions, or blocks freed while still in use")


if __name__ == "__main__":
    _try("the whole engine", _show_engine, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4",
                                                    "exercise 5", "exercise 6"))

## 11. Exercise 7: memory lost to fragmentation, counted exactly

"Memory lost to fragmentation" needs a definition before it can be measured. Count in token
slots. At any iteration, a sequence has `length` positions really filled, and the cache is **holding**
some number of slots for it. The difference is memory a running request holds and does not use:

- **paged**: the sequence holds whole blocks, so it holds `blocks_needed(length)` blocks of
  `block_size` slots. What it wastes is the unfilled tail of its last block: less than one block, always.
- **contiguous**: the sequence holds the whole slab reserved for it when it was admitted, sized for
  its prompt plus its declared cap. What it wastes is everything reserved beyond what it has
  filled *yet*: at first almost everything.

Write both as functions of the list of lengths of the running sequences, each returning
`(held, used)`: the slots held in total, and the slots actually filled in total. Section 12
checks your formulas against what the real allocators held, iteration by iteration.

<details><summary>💡 Hint 1 — what to think about</summary>

For the paged cache, which of your earlier functions says how many blocks one sequence holds? For a
sequence of length zero, how much does it hold? For the contiguous cache, what is held does not depend on
the current length at all: on what does it depend? What would it mean for a length to exceed its slab?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Sum over the sequences: for the paged cache, blocks times block size for the held slots, lengths for the
used ones. For the contiguous cache, the reserved sizes are the held slots. Refuse, with ValueError,
lists of unequal length and any sequence longer than what was reserved for it.
</details>

In [ ]:
def paged_slots(lengths: Sequence[int], block_size: int) -> tuple[int, int]:
    """(held, used) token slots of a paged cache for sequences with these `lengths` (filled positions):
    each sequence holds `blocks_needed(length, block_size)` whole blocks. Both plain ints.

    Worked example: block_size 4, lengths [1, 4, 5, 0] -> held 4 + 4 + 8 + 0 = 16, used 10.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def contiguous_slots(lengths: Sequence[int], reserved: Sequence[int]) -> tuple[int, int]:
    """(held, used) token slots of a contiguous cache: each sequence holds the whole slab reserved
    for it, `reserved[i]` slots, whatever its current length. Raises ValueError if the two lists differ in
    length, or any sequence is longer than its slab.

    Worked example: lengths [1, 4, 5], reserved [10, 4, 8] -> held 22, used 10. Lengths [5] with
    reserved [4] raises ValueError: the sequence would not fit its slab.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_slots() -> None:
    got = paged_slots([1, 4, 5, 0], 4)
    assert got == (16, 10), f"paged_slots([1, 4, 5, 0], 4) gave {got}, expected (16, 10): whole blocks, and an empty sequence holds none"
    assert paged_slots([], 4) == (0, 0), "no sequences hold nothing"
    for n in (1, 2, 3, 4, 5, 7, 8, 9):
        held, used = paged_slots([n], 4)
        assert held - used < 4 and held >= used and held % 4 == 0, (
            f"a sequence of {n} positions: held {held}, used {used}. Paged waste is always less than one block")
    assert contiguous_slots([1, 4, 5], [10, 4, 8]) == (22, 10), (
        "contiguous_slots([1, 4, 5], [10, 4, 8]) should be (22, 10): held is the SUM OF THE RESERVATIONS")
    for lengths, reserved in (([5], [4]), ([1, 2], [3])):
        try:
            contiguous_slots(lengths, reserved)
        except ValueError:
            continue
        raise AssertionError(f"contiguous_slots({lengths}, {reserved}) should raise ValueError")
    assert contiguous_slots([4], [4]) == (4, 4), "a sequence that exactly fills its slab is fine"
    assert all(type(x) is int for x in paged_slots([3], 4) + contiguous_slots([3], [4])), "return plain ints"
    print("paged_slots and contiguous_slots: whole blocks, the tail bound, reservations, and both error cases")


if __name__ == "__main__":
    _try("exercise 7", _check_slots, needs=("exercise 1",))

## 12. Three ways to serve the same trace

The engine served a handful of real requests. To compare designs you need many more, and the model is not
needed for that: a scheduler and a block table depend on lengths, not on logits. So the rest of
the lesson runs **the same scheduler code** on a trace of requests with no model attached, in
logical iterations, and compares three servers on **one trace and one memory budget**:

- **static + contiguous**: T05-L01's server, with one change. T05-L01 cut the requests into fixed groups
  of `batch_size` in arrival order and had no memory budget. Here a batch must fit the budget, so it is
  formed differently: the head of the line plus whoever has already arrived behind it, while the batch's
  rectangular slab (every row as long as the longest prompt plus the longest cap) still fits. The batch
  then runs until its longest request is done, as in T05-L01. Given.
- **continuous + contiguous**: iteration-level scheduling, but each request still reserves one slab,
  sized for prompt plus cap, from a first-fit allocator over the whole budget. Given.
- **continuous + paged**: your block pool, tables and scheduler, allocating blocks as tokens
  appear. It uses `plan_iteration`, so exercises 1 to 6 run here.

The budget is the same number of token slots for all three. Time is in iterations, not seconds: an
iteration over many requests costs more real time than one over few, and this comparison does not
know that. It also charges nothing for recomputation: an evicted request's rebuild, however long, rides
in one iteration like any prefill, which flatters the paged server. So each table prints how many
positions were prefilled again. Read every table below as "in logical steps, on this trace", nothing more.

In [ ]:
class SlabAllocator:
    """First-fit allocator of contiguous ranges of slots, with neighbouring holes merged. What a
    contiguous cache needs, and what T05-L01's per-batch slab avoided by never sharing. Given."""

    def __init__(self, size: int):
        self.size = size
        self.holes = [(0, size)]         # (start, length), sorted, never adjacent

    def alloc(self, n: int):
        """Start of a free range of `n` slots, or None if no single hole is big enough."""
        for i, (start, length) in enumerate(self.holes):
            if length >= n:
                if length == n:
                    del self.holes[i]
                else:
                    self.holes[i] = (start + n, length - n)
                return start
        return None

    def free(self, start: int, n: int) -> None:
        merged: list[tuple[int, int]] = []
        for s, length in sorted(self.holes + [(start, n)]):
            if merged and merged[-1][0] + merged[-1][1] == s:
                merged[-1] = (merged[-1][0], merged[-1][1] + length)
            else:
                merged.append((s, length))
        self.holes = merged

    @property
    def free_total(self) -> int:
        return sum(length for _, length in self.holes)

    @property
    def largest_hole(self) -> int:
        return max((length for _, length in self.holes), default=0)


class Outcome(NamedTuple):
    """What one server did with one trace. Times are iterations; slots are token slots."""
    name: str
    makespan: int          # iteration at which the last token appeared
    tokens: int            # tokens produced
    mean_ttft: float       # mean of (first token time - arrival)
    worst_ttft: int
    mean_running: float    # requests making progress, averaged over the iterations in which anything ran
    busy: int              # iterations in which anything ran
    held: int              # slots held, summed over those iterations
    used: int              # slots that held a running request's keys and values, summed likewise
    preemptions: int
    recomputed: int        # positions prefilled again after evictions
    refused_with_room: int  # iterations where the head of the line was refused although enough slots were free IN TOTAL
    series: list           # per iteration: (measured slots held, lengths of running requests, reserved sizes or None)


def _requests(trace) -> list:
    return [Request(e.rid, e.arrival, e.prompt_len, e.new_tokens, e.cap) for e in trace]


def _outcome(name, reqs, busy, held, used, running_sum, refused, series, tokens=None) -> Outcome:
    ttft = [r.first_token_at - r.arrival for r in reqs]
    return Outcome(name, max(r.finish_at for r in reqs), sum(r._new_tokens for r in reqs),
                   sum(ttft) / len(ttft), max(ttft), running_sum / busy, busy, held, used,
                   sum(r.preemptions for r in reqs), sum(r.recomputed for r in reqs), refused, series)


def _reservation(e) -> int:
    """Slots a contiguous cache must reserve for a request: its prompt and every token it might produce.
    The last token is never fed back, so it is never cached."""
    return e.prompt_len + e.cap - 1


def simulate_static(trace, pool_slots: int, max_batch: int) -> Outcome:
    """T05-L01's server on a trace. Whenever the server is free, take the request at the head of the line
    (waiting for it if it has not arrived) and every request behind it that has already arrived, in order,
    while the batch has at most `max_batch` rows and its rectangular slab (rows x (longest prompt + longest
    cap - 1)) fits `pool_slots`. The batch runs until its longest request is done. Given."""
    reqs = _requests(trace)
    n = len(reqs)
    free_at = i = busy = held = used = running_sum = 0
    while i < n:
        start = max(free_at, reqs[i].arrival)
        members = [i]
        while i + len(members) < n and len(members) < max_batch and reqs[i + len(members)].arrival <= start:
            cand = members + [i + len(members)]
            slab = max(reqs[m].prompt_len for m in cand) + max(reqs[m].cap for m in cand) - 1
            if len(cand) * slab > pool_slots:
                break
            members = cand
        slab = max(reqs[m].prompt_len for m in members) + max(reqs[m].cap for m in members) - 1
        if len(members) * slab > pool_slots:
            raise ValueError("a single request does not fit the pool")
        longest = max(reqs[m]._new_tokens for m in members)
        for step in range(longest):
            alive = [m for m in members if reqs[m]._new_tokens > step]
            held += len(members) * slab
            used += sum(reqs[m].prompt_len + step for m in alive)
            running_sum += len(alive)
        for m in members:
            reqs[m].first_token_at = start + 1
            reqs[m].finish_at = start + reqs[m]._new_tokens
        busy += longest
        free_at = start + longest
        i += len(members)
    return _outcome("static + contiguous", reqs, busy, held, used, running_sum, 0, [])


def simulate_contiguous(trace, pool_slots: int, max_running: int) -> Outcome:
    """Continuous batching with one contiguous slab per request, from a first-fit allocator. First come first
    served: a head of the line that finds no hole big enough holds everybody up. Given."""
    reqs = _requests(trace)
    if any(_reservation(e) > pool_slots for e in trace):
        raise ValueError("a single request does not fit the pool")
    alloc, where = SlabAllocator(pool_slots), {}
    waiting, running, series = list(reqs), [], []
    now = busy = held = used = running_sum = refused_with_room = 0
    while waiting or running:
        if not running and waiting[0].arrival > now:
            now = waiting[0].arrival
        refused = False
        while waiting and len(running) < max_running and waiting[0].arrival <= now:
            head = waiting[0]
            size = head.prompt_len + head.cap - 1
            start = alloc.alloc(size)
            if start is None:
                refused = alloc.free_total >= size          # room in total, but not in one piece
                break
            where[head.rid] = (start, size)
            running.append(waiting.pop(0))
        refused_with_room += int(refused)
        lengths = [r.prompt_len + r.generated for r in running]
        reserved = [where[r.rid][1] for r in running]
        series.append((sum(reserved), lengths, reserved))
        busy += 1
        held += sum(reserved)
        used += sum(lengths)
        running_sum += len(running)
        for r in list(running):
            r.generated += 1
            if r.first_token_at is None:
                r.first_token_at = now + 1
            if r.generated == r._new_tokens:
                r.finish_at = now + 1
                alloc.free(*where[r.rid])
                running.remove(r)
        now += 1
    return _outcome("continuous + contiguous", reqs, busy, held, used, running_sum, refused_with_room, series)


def simulate_paged(trace, pool_slots: int, block_size: int, max_running: int) -> Outcome:
    """Continuous batching with a paged cache: YOUR scheduler and block pool, no model. Given."""
    reqs = _requests(trace)
    state = SchedState(reqs, pool_slots // block_size, block_size, max_running)
    series = []
    now = busy = held = used = running_sum = refused_with_room = 0
    limit = 8 * sum(e.new_tokens for e in trace) + max(e.arrival for e in trace) + 1000
    while state.waiting or state.running:
        if not state.running and state.waiting[0].arrival > now:
            now = state.waiting[0].arrival
        evicted, admitted = plan_iteration(state, now)
        _ensure_progress(state, now, evicted)
        head = state.waiting[0] if state.waiting else None
        if (head is not None and not evicted and len(state.running) < max_running and head.arrival <= now
                and blocks_needed(head.prompt_len + head.generated, block_size) <= state.pool.n_free):
            refused_with_room += 1                          # room in the pool, yet the head was not admitted
        lengths = [r.prompt_len + r.generated for r in state.running]
        measured = sum(len(r.table) for r in state.running) * block_size
        series.append((measured, lengths, None))
        busy += 1
        held += measured
        used += sum(lengths)
        running_sum += len(state.running)
        advance(state, now)
        now += 1
        if busy > limit:
            raise AssertionError(f"still running after {busy} iterations: requests are evicted and readmitted "
                                 "without finishing (check WHOM grow_or_preempt evicts)")
    return _outcome("continuous + paged", reqs, busy, held, used, running_sum, refused_with_room, series)


TRACE_N, POOL_SLOTS, BLOCK_SIZE, MAX_RUNNING = 400, 3072, 16, 64


def _throughput(o: Outcome) -> float:
    return o.tokens / o.makespan


def _table(outcomes: Sequence[Outcome]) -> None:
    print(f"{'server':<24} {'makespan':>8} {'tokens/iter':>11} {'mean TTFT':>9} {'worst TTFT':>10} "
          f"{'running':>7} {'used/held':>9} {'evictions':>9} {'refused w/ room':>15}")
    for o in outcomes:
        print(f"{o.name:<24} {o.makespan:>8} {_throughput(o):>11.2f} {o.mean_ttft:>9.1f} {o.worst_ttft:>10} "
              f"{o.mean_running:>7.1f} {100 * o.used / o.held:>8.1f}% {o.preemptions:>9} {o.refused_with_room:>15}")


def _three(trace, pool_slots=POOL_SLOTS, block_size=BLOCK_SIZE) -> list:
    return [simulate_static(trace, pool_slots, MAX_RUNNING),
            simulate_contiguous(trace, pool_slots, MAX_RUNNING),
            simulate_paged(trace, pool_slots, block_size, MAX_RUNNING)]


def _show_accounting() -> None:
    trace = make_trace(TRACE_N, SEED + 10)
    paged = simulate_paged(trace, POOL_SLOTS, BLOCK_SIZE, MAX_RUNNING)
    contig = simulate_contiguous(trace, POOL_SLOTS, MAX_RUNNING)
    ok_paged = sum(paged_slots(lengths, BLOCK_SIZE)[0] == measured for measured, lengths, _ in paged.series)
    ok_contig = sum(contiguous_slots(lengths, reserved)[0] == measured for measured, lengths, reserved in contig.series)
    worst = max((measured - sum(lengths)) / len(lengths) for measured, lengths, _ in paged.series)
    print(f"a trace of {len(trace)} requests, a budget of {POOL_SLOTS} slots, blocks of {BLOCK_SIZE}")
    print(f"paged_slots equals the slots the block pool really held on {ok_paged} of {len(paged.series)} iterations")
    print(f"contiguous_slots equals the slots the first-fit allocator really held on {ok_contig} of {len(contig.series)}")
    print(f"the largest per-request average, over the iterations, of slots held beyond what was filled: {worst:.2f}, "
          f"{'below' if worst < BLOCK_SIZE else 'NOT below'} the block size of {BLOCK_SIZE}")
    assert ok_paged == len(paged.series) and ok_contig == len(contig.series), (
        "your formulas disagree with the allocators on some iterations: paged sequences hold WHOLE blocks "
        "(blocks_needed of the length, times block_size); contiguous ones hold their reservation")


if __name__ == "__main__":
    _try("accounting matches the allocators", _show_accounting,
         needs=("exercise 1", "exercise 2", "exercise 5", "exercise 6", "exercise 7"))

### The comparison

One trace: requests arrive in a steady stream, prompts and caps are uniform over ranges, and
each request really produces a uniformly random number of tokens up to its cap, because real
requests stop early (`make_trace`'s docstring says exactly what is drawn). All three servers get the
same budget of token slots and the same limit on running requests. Columns: **tokens/iter** is
tokens produced per iteration of the whole run; **TTFT** is time to first token in iterations;
**running** is the mean number of requests making progress in an iteration; **used/held** is the
share of the slots the server held that really held keys and values; **refused w/ room** counts
iterations in which the head of the line was turned away although enough slots were free in total,
just not in one piece.

In [ ]:
def _show_comparison() -> None:
    trace = make_trace(TRACE_N, SEED + 10)
    outcomes = _three(trace)
    print(f"{len(trace)} requests, {sum(e.new_tokens for e in trace)} tokens to produce, {POOL_SLOTS} slots of budget "
          f"for each server, arrivals over {trace[-1].arrival} iterations")
    _table(outcomes)
    static, contig, paged = outcomes
    print(f"continuous batching alone (contiguous slabs): {_throughput(contig) / _throughput(static):.2f} times "
          f"static's throughput and {contig.mean_ttft / static.mean_ttft:.2f} times its mean time to first token")
    print(f"paging on top: {_throughput(paged) / _throughput(contig):.2f} times the throughput of continuous "
          f"contiguous, {paged.mean_ttft / contig.mean_ttft:.2f} times its mean time to first token; "
          f"it held {100 * paged.held / (paged.busy * POOL_SLOTS):.0f}% of the budget on average against "
          f"{100 * contig.held / (contig.busy * POOL_SLOTS):.0f}%")
    print(f"paged: {paged.preemptions} evictions, {paged.recomputed} positions recomputed "
          f"({100 * paged.recomputed / paged.tokens:.1f}% of the tokens produced)")


if __name__ == "__main__":
    _try("three servers", _show_comparison, needs=("exercise 1", "exercise 2", "exercise 5", "exercise 6"))

## 13. When does paging help, and by how much?

One table does not say when a result holds. Three questions, each a sweep on the same trace
machinery, each printing what happened rather than what the previous table led you to expect:

1. **How big is the budget?** What happens to each server when memory is plentiful, and when it is
   tight enough that the paged server has to evict?
2. **How big is a block?** A bigger block wastes more of each sequence's last block and needs fewer table
   entries. Table entries are metadata a real kernel must read on every step. Where is the trade?
3. **Do requests really stop early?** Paging avoids reserving room for tokens that are never produced, and for
   tokens that are not produced yet. If every request uses its whole cap, only the second kind of waste is
   left: is any advantage left?

In [ ]:
def _show_pool_sweep() -> None:
    trace = make_trace(TRACE_N, SEED + 10)
    print("tokens produced per iteration, by server, as the memory budget grows")
    print(f"{'budget (slots)':>14} {'static':>8} {'cont+contig':>12} {'cont+paged':>11} {'paged evictions':>16} {'paged / contig':>15}")
    wins = 0
    sizes = (512, 1024, 2048, 4096, 8192)
    for slots in sizes:
        static, contig, paged = _three(trace, pool_slots=slots)
        wins += int(_throughput(paged) > _throughput(contig))
        print(f"{slots:>14} {_throughput(static):>8.2f} {_throughput(contig):>12.2f} {_throughput(paged):>11.2f} "
              f"{paged.preemptions:>16} {_throughput(paged) / _throughput(contig):>15.2f}")
    print(f"paged beat continuous+contiguous on throughput at {wins} of {len(sizes)} budgets; where it did not, "
          "the ratio above is 1.00 or lower")


def _show_block_size_sweep() -> None:
    trace = make_trace(TRACE_N, SEED + 10)
    print(f"{'block size':>10} {'slots wasted per running request':>32} {'(b - 1) / 2':>11} {'table entries per request':>26} "
          f"{'tokens/iter':>11} {'evictions':>9}")
    wasted_col, entries_col = [], []
    for block_size in (2, 4, 8, 16, 32, 64):
        o = simulate_paged(trace, POOL_SLOTS, block_size, MAX_RUNNING)
        request_iterations = sum(len(lengths) for _, lengths, _ in o.series)
        wasted = (o.held - o.used) / request_iterations
        entries = sum(measured // block_size for measured, _, _ in o.series) / request_iterations
        wasted_col.append(wasted)
        entries_col.append(entries)
        print(f"{block_size:>10} {wasted:>32.2f} {(block_size - 1) / 2:>11.2f} {entries:>26.2f} {_throughput(o):>11.2f} {o.preemptions:>9}")
    print(f"waste per request rose at every step up in block size: {all(a < b for a, b in zip(wasted_col, wasted_col[1:]))}; "
          f"table entries per request fell at every step: {all(a > b for a, b in zip(entries_col, entries_col[1:]))}. "
          "The middle column is what you would expect if the fill of a last block were equally likely to be any of "
          "the block's slots.")


def _show_slack() -> None:
    summary = []
    for label, kwargs in (("requests stop early", {}), ("every request runs to its cap", {"run_to_cap": True})):
        trace = make_trace(TRACE_N, SEED + 10, **kwargs)
        print(f"{label}: {sum(e.new_tokens for e in trace)} tokens to produce for {sum(e.cap for e in trace)} declared")
        outcomes = _three(trace)
        _table(outcomes)
        summary.append((label, outcomes))
    for label, (static, contig, paged) in summary:
        print(f"{label}: paged / contiguous throughput {_throughput(paged) / _throughput(contig):.2f}; "
              f"contiguous kept {100 * contig.used / contig.held:.1f}% of its held slots in use and refused the head of "
              f"the line with room in total on {contig.refused_with_room} of {contig.busy} iterations")


if __name__ == "__main__":
    _try("budget sweep", _show_pool_sweep, needs=("exercise 1", "exercise 2", "exercise 5", "exercise 6"))
    _try("block size sweep", _show_block_size_sweep, needs=("exercise 1", "exercise 2", "exercise 5", "exercise 6"))
    _try("stopping early", _show_slack, needs=("exercise 1", "exercise 2", "exercise 5", "exercise 6"))

## 14. Common mistakes

- **Block id 0 is a block.** `if block:` and `table[i] or default` treat it as "nothing". The pool
  hands out the lowest ids first, so block 0 is the first one used.
- **A negative position is not "from the end".** `table[-1]` quietly reads the last block. A position
  outside the table is an error.
- **Gathering after the fact.** Write the new key and value, THEN gather, or the new token cannot attend
  to itself. Gather exactly `length` positions: the rest of a block belongs to nobody, or to a
  previous tenant.
- **Freeing without forgetting.** A request whose blocks went back to the pool but whose table still
  lists them will release them a second time when it finishes, and two requests will share one block.
- **Serving a request you have just evicted.** If you walk a copy of `state.running`, a request evicted
  earlier in the pass is still in your copy: skip it, or it is handed blocks while it waits.
- **Evicting the wrong request.** Evicting the oldest looks harmless in a small test and lets the request that
  has waited longest lose its work again and again.
- **Forgetting progress on readmission.** An evicted request must rebuild its prompt AND the tokens it
  already produced; sizing its blocks from the prompt alone leaves it short at the first decode step.
- **Skipping the head of the line.** Admitting a small request behind one that does not fit starves the
  big one. First come, first served is a policy, and here it is the tested one.
- **Comparing throughput in iterations as if they were seconds.** An iteration over many requests
  costs more than one over few. Say which unit a number is in.

In [ ]:
# The two Python facts behind the first two mistakes, computed rather than asserted.
_demo_table = [7, 0, 9]
print(f"table[-1] is {_demo_table[-1]}, the last block, not an error; and bool(0) is {bool(_demo_table[1])}, so `if block:` would treat block 0 as missing")

## 15. Self-check

1. A paged cache holds a sequence of `n` positions in blocks of `block_size` slots. The most that
   sequence can hold beyond what it has filled is:
   - (a) nothing: paging is exact
   - (b) half of `n`
   - (c) less than one block, the unfilled tail of its last block
   - (d) the difference between its cap and its length

2. Why can a paged server never turn a request away while enough free blocks exist in total, when a
   contiguous server can?
   - (a) Its blocks are larger than the slabs
   - (b) Every block is the same size, so any free block serves any request and no free space has to be
     adjacent to anything else
   - (c) It moves running requests around in memory every iteration to close the gaps
   - (d) It never allocates more than the prompt needs

3. A request is evicted by recomputation and later readmitted. Why does it go on to produce exactly the
   tokens it would have produced without the eviction?
   - (a) Its old blocks were kept aside and are given back to it
   - (b) Nothing about its output depends on the cache, only on its prompt
   - (c) The scheduler stores its scores and replays them
   - (d) A position's keys and values depend only on the tokens up to it, so prefilling the prompt plus the
     tokens it already produced rebuilds the same cache, and its next token is drawn from the same scores
     with the same random number

4. When the pool is short, the scheduler evicts the most recently admitted request, not the oldest.
   The main reason is:
   - (a) it keeps first come, first served: the requests that have waited longest keep the progress
     they have made, and the one with the least sunk work pays
   - (b) the newest request always holds the most blocks
   - (c) evicting the oldest is impossible, because its blocks cannot be freed
   - (d) the newest request is guaranteed to be the shortest

5. This lesson reports throughput as tokens per iteration. What does that number leave out?
   - (a) Nothing: it is the same as tokens per second
   - (b) The memory the server used
   - (c) How long an iteration takes, which grows with the number of requests in it and depends on the
     machine, so it compares schedules and not hardware
   - (d) The tokens that were recomputed after an eviction

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
print("self-check: work the questions above, then compare with the worked solution")

## 16. What this lesson does not cover

This lesson keeps everything at toy scale so that you can read all of it. It serves one layer and one
head, so a "block" is one array; a real engine's block holds keys and values for every layer and head, and
an attention kernel reads the block table on the GPU and never copies a gathered array, as `paged_gather`
does. It never splits a long prefill across iterations (chunked prefill), so a long prompt stalls the
decoding of everybody else in its iteration; it never shares blocks between requests that begin with the
same text (prefix caching, which needs reference counts and copy-on-write); it never swaps a preempted
request's blocks to CPU memory, and it holds no spare blocks back to make eviction rarer. As of
2026-10-14 the newest vLLM release is v0.31.0, and its documentation says that in vLLM V1 the default
preemption mode is recompute rather than swap, the mode you wrote (`claims.yaml` has the sources).
It measures logical iterations, not seconds, so it says nothing about how fast any of this runs on a
GPU, and it uses one fixed trace family whose assumptions (uniform lengths, an early-stop rule) you
saw stated in `make_trace`.

## What you built

A block pool, a block table, and a paged cache that gives back the same keys and values a slab does; a
ragged batched decode step; a continuous-batching scheduler with first-come-first-served admission and
preemption by recomputation, proved on real text to change nothing a request says; and the exact
accounting of what a contiguous cache and a paged cache each lose. The habit to keep: build the fast
path, then prove it equals the slow one on the same inputs before you believe any speed-up.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now: each one already printed its feedback in its own cell above, so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in (
                ("exercise 1", _check_blocks_and_locate, ()),
                ("exercise 2", _check_pool, ()),
                ("exercise 3", _check_paged_write_gather, ("exercise 1",)),
                ("exercise 4", _check_decode_paged, ("exercise 1", "exercise 3")),
                ("exercise 5", _check_grow_or_preempt, ("exercise 1", "exercise 2")),
                ("exercise 6", _check_admit, ("exercise 1", "exercise 2")),
                ("exercise 7", _check_slots, ("exercise 1",))):
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.